In [ ]:
import duckdb

In [16]:
# # Check for nulls in LDA file
# LDA_PATH = '../data/processed/lda_features.parquet'
#
# duckdb.sql(f"""
#     SELECT
#         COUNT(*) AS total_rows,
#         COUNT("dominant_topic") AS non_null_dominant_topic,
#         COUNT(*) - COUNT("dominant_topic") AS null_dominant_topic
#     FROM '{LDA_PATH}'
# """).df()

,total_rows,non_null_dominant_topic,null_dominant_topic
0,398004,398004,0


In [17]:
# Join the K-Means and LDA features and export to file
LEFT_PATH = '../data/processed/clustered_narratives.parquet'
LDA_PATH = '../data/processed/lda_features.parquet'
OUTPUT_PATH = '../data/processed/final_unsupervised_features.parquet'

duckdb.sql(f"""
    COPY (
        SELECT
            l.*,
            r.* EXCLUDE ("Complaint ID")
        FROM '{LEFT_PATH}' AS l
        LEFT JOIN '{LDA_PATH}' AS r
            ON l."Complaint ID" = r."Complaint ID"
    )
    TO '{OUTPUT_PATH}'
    (FORMAT PARQUET)
""")

In [18]:
# Record count of the combined file
COMBINED_PATH = '../data/processed/final_unsupervised_features.parquet'

# Full row count
row_count = duckdb.sql(f"""
    SELECT COUNT(*) AS total_rows
    FROM '{COMBINED_PATH}'
""").fetchone()[0]

print(f'Total rows: {row_count:,}')

Total rows: 398,004


In [19]:
# View the columns
duckdb.sql(f"""
    DESCRIBE SELECT *
    FROM '{COMBINED_PATH}'
""").df()

,column_name,column_type,null,key,default,extra
0,Date received,TIMESTAMP,YES,None,None,None
1,Product,VARCHAR,YES,None,None,None
2,Sub-product,VARCHAR,YES,None,None,None
3,Issue,VARCHAR,YES,None,None,None
4,Sub-issue,VARCHAR,YES,None,None,None
5,Consumer complaint narrative,VARCHAR,YES,None,None,None
6,Company public response,VARCHAR,YES,None,None,None
7,Company,VARCHAR,YES,None,None,None
8,State,VARCHAR,YES,None,None,None
9,ZIP code,VARCHAR,YES,None,None,None


In [20]:
# First 10 rows for selected columns
preview_df = duckdb.sql(f"""
    SELECT
        "Complaint ID",
        Product,
        Issue,
        cluster,
        distance_to_centroid,
    FROM '{COMBINED_PATH}'
    LIMIT 10
""").df()

preview_df

,Complaint ID,Product,Issue,cluster,distance_to_centroid
0,3442136,Credit card or prepaid card,Problem with a purchase shown on your statement,11,0.992864
1,3601853,Credit card or prepaid card,Trouble using the card,3,0.998468
2,3300820,Credit card or prepaid card,Problem with a purchase shown on your statement,44,0.978543
3,3739698,Credit card or prepaid card,Trouble using your card,25,0.852135
4,3285243,Credit card or prepaid card,"Advertising and marketing, including promotion...",44,0.963603
5,8873634,Checking or savings account,Problem with a lender or other company chargin...,25,0.902547
6,3739701,Checking or savings account,Managing an account,49,0.890691
7,7942358,Credit card,Problem with a purchase shown on your statement,52,0.904790
8,3589185,Checking or savings account,Managing an account,49,0.941633
9,3235193,Checking or savings account,Problem with a lender or other company chargin...,36,0.822457


In [21]:
# # Check for nulls in dominant_topic
# INPUT_PATH = '../data/processed/final_unsupervised_features.parquet'
#
# duckdb.sql(f"""
#     SELECT
#         COUNT(*) AS total_rows,
#         COUNT(dominant_topic) AS non_null_dominant_topic,
#         COUNT(*) - COUNT(dominant_topic) AS null_dominant_topic
#     FROM '{INPUT_PATH}'
# """).df()

,total_rows,non_null_dominant_topic,null_dominant_topic
0,398004,398004,0


In [22]:
# # Export CSV to sanity-check clusters
# INPUT_PATH = '../data/processed/final_unsupervised_features.parquet'
# OUTPUT_PATH = '../data/processed/complaint_clusters.csv'
#
# duckdb.sql(f"""
#     COPY (
#         SELECT
#             "Complaint ID",
#             cluster
#         FROM '{INPUT_PATH}'
#     )
#     TO '{OUTPUT_PATH}'
#     (FORMAT CSV, HEADER)
# """)

In [24]:
# # Export CSV to sanity-check dominant topic and topic_prob
# INPUT_PATH = '../data/processed/final_unsupervised_features.parquet'
# OUTPUT_PATH = '../data/processed/dominant_topic.csv'
#
# duckdb.sql(f"""
#     COPY (
#         SELECT
#             "Complaint ID",
#             dominant_topic,
#             topic_3_prob,
#         FROM '{INPUT_PATH}'
#     )
#     TO '{OUTPUT_PATH}'
#     (FORMAT CSV, HEADER)
# """)